In [ ]:
# Installations (Comment out if already installed)
!pip install qiskit
!pip install qiskit_aer
!pip install qiskit_ibm_runtime
!pip install pylatexenc

In [ ]:
# Imports
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit, QuantumRegister, qpy
from qiskit.transpiler import StagedPassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeTorino

import os

In [ ]:
def run_transpiler(file_path, backend, initial_layout=None, start_opt_lvl=3,
                   end_opt_lvl=3, iterations=200, save_image=False):
  """
  Runs the circuit from the given file through the transpiler.

  Args:
    file_path (str): The file path (either .qpy or .qasm) containing the circuit.
    backend (AerSimulator): The backend for transpilation.
    initial_layout (list of int): The initial layout for transpilation.
    start_opt_lvl (int): The optimization level for the initialization, layout, and routing stages.
    end_opt_lvl (int): The optimization level for the translation, optimization, and scheduling stages.
    iterations (int): The number of iterations to run the transpiler
    save_image (bool): Whether or not to save an image of the transpiled circuit as an .svg and .png.

  Returns:
    saved_transpiled_qc (QuantumCircuit): the final transpiled circuit
    original_swaps (int): the number of SWAP gates in the original circuit
    saved_swap_count (int): the number of SWAP gates in the circuit after routing
    saved_tqc (int): the Transpilation Quantum Cost (TQC) of the transpiled circuit
  """
  # Get circuit from file (either .qpy or .qasm)
  if os.path.splitext(file_path)[1] == ".qpy":
    with open(file_path, 'rb') as fd:
        circuits = qpy.load(fd)
    qc = circuits[0].decompose()
    original_swaps = qc.count_ops().get('swap', 0)  # Extract original number of SWAP gates
  elif os.path.splitext(file_path)[1] == ".qasm":
    qc = QuantumCircuit.from_qasm_file(file_path).decompose()
    original_swaps = qc.count_ops().get('swap', 0)  # Extract original number of SWAP gates
  else:
    print(f"Unsupported file type: {os.path.splitext(file_path)[1]}")
    return

  # Initialize variables
  saved_tqc = None
  saved_swap_count = None
  saved_transpiled_qc = None

  # Loop and save the circuit with the lowest TQC
  for _ in range(iterations):
    present_pass_manager = generate_preset_pass_manager(optimization_level=start_opt_lvl,
                                                        basis_gates=backend.target.operation_names,
                                                        coupling_map=backend.coupling_map if initial_layout is None else backend.coupling_map.reduce(initial_layout[:qc.num_qubits]))

    # Initialization, layout, and routing
    routing_manager = StagedPassManager(
        stages=["init", "layout", "routing"],
        init=present_pass_manager.init,
        layout=present_pass_manager.layout,
        routing=present_pass_manager.routing
    )
    routed_qc = routing_manager.run(qc)

    # Extract the number of SWAP gates
    swap_count = routed_qc.count_ops().get('swap', 0)

    present_pass_manager = generate_preset_pass_manager(optimization_level=end_opt_lvl,
                                                        basis_gates=backend.target.operation_names,
                                                        coupling_map=backend.coupling_map if initial_layout is None else backend.coupling_map.reduce(initial_layout[:qc.num_qubits]))

    # Translation, optimization, and scheduling
    optimization_manager = StagedPassManager(
        stages=["translation", "optimization", "scheduling"],
        translation=present_pass_manager.translation,
        optimization=present_pass_manager.optimization,
        scheduling=present_pass_manager.scheduling
    )
    expanded_transpiled_qc = optimization_manager.run(routed_qc)

    # Remove idle qubits
    used_qubits = {q for inst in expanded_transpiled_qc.data for q in inst.qubits}
    active_qubits = [q for q in expanded_transpiled_qc.qubits if q in used_qubits]
    transpiled_qc = QuantumCircuit(active_qubits, *expanded_transpiled_qc.cregs)
    transpiled_qc.global_phase = expanded_transpiled_qc.global_phase
    for inst in expanded_transpiled_qc.data:
      transpiled_qc.append(inst.operation, inst.qubits, inst.clbits)

    tqc = transpiled_qc.depth() + transpiled_qc.size() + swap_count

    # Update if TQC is lower
    if (saved_tqc is None) or (tqc < saved_tqc):
      saved_tqc = tqc
      saved_swap_count = swap_count
      saved_transpiled_qc = transpiled_qc


  # Save transpiled circuit
  with open(f"{os.path.splitext(file_path)[0]}-transpiled.qpy", "wb") as f:
    qpy.dump(saved_transpiled_qc, f)

  if save_image:
    transpiled_circuit_fig = saved_transpiled_qc.draw(output='mpl')
    transpiled_circuit_fig.savefig(f'{os.path.splitext(file_path)[0]}-transpiled.svg')
    transpiled_circuit_fig.savefig(f'{os.path.splitext(file_path)[0]}-transpiled.png')

  return saved_transpiled_qc, original_swaps, saved_swap_count, saved_tqc

# Run on Benchmarks
---

In [ ]:
# Get backend
# -----
# This can be replaced with any backend of one's choosing. IBM Torino is used here.
backend = AerSimulator.from_backend(FakeTorino())

In [ ]:
# Specify layout
# Or set to None so the routing is done by an algorithm
# -----
# This layout is based on the qubits of the IBM Torino layout
# This can be replaced with different qubits to specify a different routing of
# the virtual/logical to physical qubits.
initial_layout = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 18,
                  31, 32, 33, 37, 52, 51, 50, 49, 48, 36, 29,
                  28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 15]

In [ ]:
# Circuit type ("lattice", "lattice-cala", "esop", or "bdd")
circuit_type = "lattice"

# File type ("qasm" or qpy"")
file_type = "qasm"

In [ ]:
# Run the transpiler on a single benchmark
benchmark = "sam_ex1" # <-- Change
file_path = f"/circuits/{circuit_type}_circuits/{benchmark}-{circuit_type}.{file_type}"

transpiled_qc, original_swaps, swap_count, tqc = run_transpiler(file_path, backend, initial_layout=initial_layout)

# Print circuit info
print(f"Gates: {transpiled_qc.size()}")
print(f"Depth: {transpiled_qc.depth()}")
print(f"Total SWAPs: {swap_count}")
print(f"TQC: {tqc}")

In [ ]:
# Run the transpiler on all benchmarks
benchmarks = ["sam_ex1", "sam_ex2", "sam_ex3", "sam_ex4", "sam_ex5",
              "rd53f1", "rd53f2", "rd73f1", "rd73f3", "rd84f1", "rd84f3", "rd84f4",
              "sym6_63", "sym9_71", "sym10_207", "co14_135"]

for benchmark in benchmarks:
  file_path = f"/circuits/{circuit_type}_circuits/{benchmark}-{circuit_type}.{file_type}"
  transpiled_qc, local_swaps, swap_count, tqc = run_transpiler(file_path, backend, initial_layout=initial_layout)

  # Print circuit info
  print(f"# {benchmark}")
  print(f"Gates: {transpiled_qc.size()}")
  print(f"Depth: {transpiled_qc.depth()}")
  print(f"Total SWAPs: {swap_count}")
  print(f"TQC: {tqc} \n")